# TrendPulse — Part 1: Python Foundations & Data Structures

**What is TrendPulse?** A mini project that turns daily YouTube "Trending" lists from several countries into an evidence-backed *content strategy briefing*. This notebook is **Part 1**: load the data, clean it with core Python, organise it with the right data structures, and label every video with transparent rule-based logic (no machine learning).

**Contents**
1. Setup & configuration
2. Loading function and `category_names`
3. Type conversion & cleaning (with justified decisions)
4. Core data structures (list of dicts, tuples, sets, dict counter, slicing, nested dict)
5. Refactor — one repeated block turned into a function
6. Variable scope demonstration
7. Loops over strings and sequences (`for`, `range()`, `while`, `break`, `continue`)
8. Rule-based classification (`classify_video`) with hand-trace
9. Summary

> **Only Pandas used:** `pd.read_csv()` and `df.to_dict("records")`. Everything after loading is core Python.

## 1. Setup & configuration

**Notebook discipline followed**
- Clear numbered headings; one idea per section.
- Configuration (paths, countries, thresholds) lives in **one** cell at the top — nothing is hard-coded further down.
- No hidden state: every variable is created by a cell above it, and the notebook was run with **Restart & Run All** before saving.
- Raw data is never modified: cleaning builds *new* dictionaries (`cleaned_records`), so re-running a cell cannot corrupt the source.

**Running on Google Colab?** Uncomment the two lines below once (replace with your repo URL) so the `data/` folder is found. Running locally or in Jupyter from inside `part1_python_foundations/`, skip them.

In [68]:
# --- Google Colab only: clone the repo so the data/ folder exists, then move into Part 1 ---
# !git clone https://github.com/Abirami-02/trendpulse.git
# %cd trendpulse/part1_python_foundations

In [69]:
import pandas as pd
from datetime import datetime

# ---------- Files & countries ----------
COUNTRIES = ["US", "GB", "IN"]                      # list[str]
DATA_DIR = "data"                                   # str
CATEGORY_FILE = f"{DATA_DIR}/categories.csv"        # str
COUNTRY_FILES = {c: f"{DATA_DIR}/{c}videos.csv" for c in COUNTRIES}   # dict[str, str]
MAX_ROWS = None                                     # None = load every row
MIN_TOTAL_ROWS = 40_000                             # project requirement

# ---------- Fields to convert ----------
NUMERIC_FIELDS = ["views", "likes", "dislikes", "comment_count"]
BOOLEAN_FIELDS = ["comments_disabled", "ratings_disabled", "video_error_or_removed"]

# ---------- Classification thresholds (justified in Section 8) ----------
VIRAL_VIEWS = 1_000_000      # int   - views needed to count as a "hit"
STEADY_VIEWS = 100_000       # int   - views needed to count as "steady"
HEALTHY_RATIO = 0.90         # float - like ratio of a well-received video
DIVISIVE_RATIO = 0.80    # float - below this, reception is split

RUN_START = datetime.now()

print("=" * 55)
print("TrendPulse - Part 1: Python Foundations")
print("=" * 55)
print(f"Countries to load : {', '.join(COUNTRIES)}  ({len(COUNTRIES)} countries)")
print(f"Data folder       : {DATA_DIR}/")
print(f"Row limit         : {'all rows' if MAX_ROWS is None else MAX_ROWS}")
print(f"Run started at    : {RUN_START:%Y-%m-%d %H:%M:%S}")

TrendPulse - Part 1: Python Foundations
Countries to load : US, GB, IN  (3 countries)
Data folder       : data/
Row limit         : all rows
Run started at    : 2026-11-21 19:37:10


## 2. Loading function and `category_names`

`load_country(path, country, max_rows=None)` is reusable: it takes **parameters**, has a **default argument** (`max_rows=None`), and **returns** a list of dictionaries. It verifies every column the later sections use, so a wrong file fails immediately with a clear message instead of a confusing error 100 cells later.

In [70]:
REQUIRED_COLUMNS = [
    "video_id", "trending_date", "title", "channel_title", "category_id",
    "publish_time", "views", "likes", "dislikes", "comment_count",
    "comments_disabled", "ratings_disabled", "video_error_or_removed",
]

def load_country(path, country, max_rows=None):
    """Read one country's CSV and return its rows as a list of dictionaries."""
    df = pd.read_csv(path, nrows=max_rows)

    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")

    records = df.to_dict("records")
    for record in records:                  # core Python: tag each row with its country
        record["country"] = country

    print(f"{country}: {len(records):,} records loaded from {path}")
    return records

In [71]:
raw_records = {}                                   # country -> list of dicts
for country in COUNTRIES:
    try:
        raw_records[country] = load_country(COUNTRY_FILES[country], country, MAX_ROWS)
    except FileNotFoundError:
        print(f"!! {COUNTRY_FILES[country]} not found. Put the CSV files in the '{DATA_DIR}/' folder.")
        raise

total_loaded = sum(len(rows) for rows in raw_records.values())
print(f"\nTotal loaded: {total_loaded:,} records from {len(raw_records)} countries")
print("Meets the 40,000-row requirement:", total_loaded >= MIN_TOTAL_ROWS)

US: 16,080 records loaded from data/USvideos.csv
GB: 15,075 records loaded from data/GBvideos.csv
IN: 15,075 records loaded from data/INvideos.csv

Total loaded: 46,230 records from 3 countries
Meets the 40,000-row requirement: True


In [72]:
# category_id -> category name, built from categories.csv (to_dict, no DataFrame iteration)
category_names = {
    row["category_id"]: row["category_name"]
    for row in pd.read_csv(CATEGORY_FILE).to_dict("records")
}

print(f"{len(category_names)} categories loaded")
for category_id, name in list(category_names.items())[:5]:
    print(f"  {category_id:>2} -> {name}")

17 categories loaded
   1 -> Film & Animation
   2 -> Autos & Vehicles
  10 -> Music
  15 -> Pets & Animals
  17 -> Sports


## 3. Type conversion & cleaning (core Python)

### Decisions and justification

| Problem | Decision | Why |
|---|---|---|
| **Blank `likes` / `comment_count`** (a few dozen rows) | Keep the record, store `None` | The video's views and dislikes are still valid. Substituting `0` would claim "nobody liked it" and distort the like ratio, so we do **not** invent values. `classify_video` sends these to *Unrated*. |
| **Bad or blank value in any other field** | Keep the record, store `None` | Dropping a whole row because one field is bad throws away good information. Failures are counted per field so the damage is visible. |
| **Negative or non-whole numbers** | Treated as *bad* (`None`) | Views, likes, dislikes and comment counts cannot be negative or fractional. |
| **Boolean flags** | Convert `"True"/"False"` text (or real booleans) to `bool`; anything else becomes `None` | An unknown flag is never silently treated as `False`. |
| **`trending_date`** | Parsed with format `%y.%d.%m` | This copy stores dates as `yy.dd.mm` (e.g. `17.14.11` = **14 Nov 2017**), checked below. |
| **`publish_time`** | Parsed ISO timestamp (`...T07:04:28.000Z`) into a timezone-aware `datetime` | Needed later for time-based analysis. |
| **Exact duplicate rows** (same country + `video_id` + `trending_date`) | **Drop** the repeats | A video can only appear once per country per day, so repeats inflate channel/category counts. This is the only case where we drop rows. |

Every conversion is wrapped in `try/except`, so one bad value cannot crash the notebook.

In [73]:
def is_blank(value):
    """True for None, NaN (how pandas stores an empty CSV cell) and empty/whitespace strings."""
    if value is None:
        return True
    if isinstance(value, float) and value != value:     # NaN is the only value not equal to itself
        return True
    return str(value).strip() == ""

def to_int(value):
    """Convert to a non-negative whole number or raise ValueError."""
    number = float(str(value).strip().replace(",", ""))
    if number < 0 or number != int(number):
        raise ValueError(f"not a non-negative whole number: {value!r}")
    return int(number)

def to_bool(value):
    """Convert True/False (or 'true'/'false' text) to bool, otherwise raise ValueError."""
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in ("true", "1", "yes"):
        return True
    if text in ("false", "0", "no"):
        return False
    raise ValueError(f"not a boolean: {value!r}")

def parse_trending_date(value):
    """'17.14.11' (yy.dd.mm) -> datetime.date(2017, 11, 14)."""
    return datetime.strptime(str(value).strip(), "%y.%d.%m").date()

def parse_publish_time(value):
    """'2017-11-14T07:04:28.000Z' -> timezone-aware datetime (UTC)."""
    return datetime.fromisoformat(str(value).strip().replace("Z", "+00:00"))

def convert_field(value, converter):
    """Apply a converter safely. Returns (converted_value, status) with status 'ok', 'blank' or 'bad'."""
    if is_blank(value):
        return None, "blank"
    try:
        return converter(value), "ok"
    except (ValueError, TypeError):
        return None, "bad"

# field -> converter function (functions are objects, so they can be dictionary values)
FIELD_CONVERTERS = {field: to_int for field in NUMERIC_FIELDS}
FIELD_CONVERTERS.update({field: to_bool for field in BOOLEAN_FIELDS})
FIELD_CONVERTERS["trending_date"] = parse_trending_date
FIELD_CONVERTERS["publish_time"] = parse_publish_time

# quick check of the date format on this copy of the data
sample = raw_records["US"][0]
print("Raw trending_date :", sample["trending_date"], "->", parse_trending_date(sample["trending_date"]))
print("Raw publish_time  :", sample["publish_time"], "->", parse_publish_time(sample["publish_time"]))

Raw trending_date : 17.14.11 -> 2017-11-14
Raw publish_time  : 2017-11-14T07:04:28.000Z -> 2017-11-14 07:04:28+00:00


In [74]:
cleaned_records = []                               # list of dictionaries (new dicts, raw data untouched)
seen_keys = set()                                  # tuples (country, video_id, trending_date) already kept
duplicates_dropped = {country: 0 for country in COUNTRIES}
failure_counts = {field: {"blank": 0, "bad": 0} for field in FIELD_CONVERTERS}

for country in COUNTRIES:
    for record in raw_records[country]:

        # --- duplicate check: a tuple is hashable, so it can live inside a set ---
        key = (record["country"], record["video_id"], record["trending_date"])
        if key in seen_keys:
            duplicates_dropped[country] += 1
            continue
        seen_keys.add(key)

        # --- convert every field on a copy of the record ---
        new_record = dict(record)
        for field, converter in FIELD_CONVERTERS.items():
            value, status = convert_field(record[field], converter)
            new_record[field] = value
            if status != "ok":
                failure_counts[field][status] += 1

        cleaned_records.append(new_record)

print("Cleaning finished.\n")
print(f"{'Field':<24}{'Blank':>8}{'Bad format':>12}")
print("-" * 44)
for field, counts in failure_counts.items():
    print(f"{field:<24}{counts['blank']:>8}{counts['bad']:>12}")

print("\nDuplicate rows dropped:", duplicates_dropped, "->", sum(duplicates_dropped.values()), "total")
print(f"Records kept: {len(cleaned_records):,} (from {total_loaded:,} loaded)")

Cleaning finished.

Field                      Blank  Bad format
--------------------------------------------
views                          0           0
likes                         12           0
dislikes                       0           0
comment_count                 36           0
comments_disabled              0           0
ratings_disabled               0           0
video_error_or_removed         0           0
trending_date                  0           0
publish_time                   0           0

Duplicate rows dropped: {'US': 80, 'GB': 75, 'IN': 75} -> 230 total
Records kept: 46,000 (from 46,230 loaded)


In [75]:
# Verify the types after cleaning
first = cleaned_records[0]
for field in ["views", "likes", "comments_disabled", "trending_date", "publish_time"]:
    print(f"{field:<18}{first[field]!r:<45}{type(first[field]).__name__}")

print()
for country in COUNTRIES:
    kept = sum(1 for r in cleaned_records if r["country"] == country)
    print(f"{country}: {kept:,} cleaned records")

views             48555                                        int
likes             2189                                         int
comments_disabled False                                        bool
trending_date     datetime.date(2017, 11, 14)                  date
publish_time      datetime.datetime(2017, 11, 14, 7, 4, 28, tzinfo=datetime.timezone.utc)datetime

US: 16,000 cleaned records
GB: 15,000 cleaned records
IN: 15,000 cleaned records


## 4. Core data structures

| Structure | Used for | Why it was the right choice | Mutable? |
|---|---|---|---|
| **List of dictionaries** (`cleaned_records`) | The cleaned dataset | A list keeps row order and can be looped; each dict gives readable named fields (`record["views"]`) instead of position numbers. | List: **mutable**, dict: **mutable** |
| **Tuple** `(video_id, trending_date, views, likes)` | A fixed snapshot of key fields; also the duplicate key `(country, video_id, trending_date)` | A snapshot should never change by accident, and only immutable (hashable) objects can be stored in a set or used as dictionary keys. | **Immutable** |
| **Set** | Unique video IDs per country | Sets discard duplicates automatically and give fast `in` checks, plus `intersection` / `difference` for "trended in several countries" questions. | **Mutable** (its items must be immutable) |
| **Dictionary counter** | Trending appearances per channel | Channel name -> count gives instant lookup and update. | **Mutable** |
| **Slicing** `[:15]` | Top 15 channels | Takes the first 15 items of a sorted list without a manual loop. | (slicing a list returns a new list) |
| **Nested dictionary** | country -> category name -> count | Two natural levels of grouping; one lookup `summary["US"]["Music"]` answers the question. | **Mutable** |

In [76]:
# Tuples: immutable snapshot of the key fields for every cleaned record
video_tuples = [
    (r["video_id"], r["trending_date"], r["views"], r["likes"])
    for r in cleaned_records
]

print(f"{len(video_tuples):,} tuples created. First 3:")
for item in video_tuples[:3]:
    print(" ", item)

# Proof that a tuple is immutable
try:
    video_tuples[0][2] = 0
except TypeError as error:
    print("\nTrying to change a tuple ->", type(error).__name__, ":", error)

46,000 tuples created. First 3:
  ('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
  ('Ugyql2NlPAd', datetime.date(2017, 11, 14), 198385, 1159)
  ('8fE19jkxJ9n', datetime.date(2017, 11, 14), 231656, 8742)

Trying to change a tuple -> TypeError : 'tuple' object does not support item assignment


In [77]:
def unique_ids(records):
    """Return the set of unique video IDs in a list of records."""
    ids = set()
    for record in records:
        ids.add(record["video_id"])
    return ids

# Records grouped by country, then a set of unique IDs per country
records_by_country = {c: [r for r in cleaned_records if r["country"] == c] for c in COUNTRIES}
ids = {c: unique_ids(records_by_country[c]) for c in COUNTRIES}

for country in COUNTRIES:
    print(f"Unique videos in {country}: {len(ids[country]):,}  (from {len(records_by_country[country]):,} trending rows)")

Unique videos in US: 5,068  (from 16,000 trending rows)
Unique videos in GB: 4,113  (from 15,000 trending rows)
Unique videos in IN: 5,878  (from 15,000 trending rows)


In [78]:
us, gb, ind = ids["US"], ids["GB"], ids["IN"]

# ---- Intersection: videos that trended in more than one country ----
print("Trended in both US & GB :", len(us & gb))
print("Trended in both US & IN :", len(us & ind))
print("Trended in both GB & IN :", len(gb & ind))
in_two_or_more = (us & gb) | (us & ind) | (gb & ind)
in_all_three = us & gb & ind
print("Trended in 2+ countries :", len(in_two_or_more))
print("Trended in all 3        :", len(in_all_three))

# ---- Difference: videos unique to one country ----
us_only = us - gb - ind
gb_only = gb - us - ind
in_only = ind - us - gb
print("\nUnique to US only:", len(us_only))
print("Unique to GB only:", len(gb_only))
print("Unique to IN only:", len(in_only))

# Sanity check: every video is either in exactly one country or in 2+
print("\nCheck (unique-to-one + in-2+ = all unique IDs):",
      len(us_only) + len(gb_only) + len(in_only) + len(in_two_or_more) == len(us | gb | ind))

Trended in both US & GB : 134
Trended in both US & IN : 120
Trended in both GB & IN : 126
Trended in 2+ countries : 340
Trended in all 3        : 20

Unique to US only: 4834
Unique to GB only: 3873
Unique to IN only: 5652

Check (unique-to-one + in-2+ = all unique IDs): True


In [79]:
# Dictionary counter: trending appearances per channel
channel_counts = {}
for record in cleaned_records:
    channel = record["channel_title"]
    if channel not in channel_counts:
        channel_counts[channel] = 0
    channel_counts[channel] += 1

print(f"{len(channel_counts)} distinct channels")

# Slicing: top 15 channels by appearances
sorted_channels = sorted(channel_counts.items(), key=lambda item: item[1], reverse=True)
top_15_channels = sorted_channels[:15]

print("\nTop 15 channels by trending appearances:")
for rank, (channel, count) in enumerate(top_15_channels, start=1):
    print(f"{rank:>2}. {channel:<28}{count:>6,}")

438 distinct channels

Top 15 channels by trending appearances:
 1. QuietChallenge75             2,451
 2. OfficialReview41             1,577
 3. UltimateUnboxing65           1,428
 4. InsaneReaction46             1,226
 5. Brand NewChallenge45         1,172
 6. CrazyTrailer91               1,012
 7. FinalTrailer17                 904
 8. UltimateSpeech20               766
 9. WildHighlights7                682
10. Brand NewSecret51              681
11. SurprisingHighlights41         648
12. FinalBreakdown11               642
13. CrazyRecipe67                  601
14. QuietInterview9                565
15. UltimateVlog96                 562


In [80]:
# Nested dictionary: country -> category name -> number of records
country_category_counts = {}
for record in cleaned_records:
    country = record["country"]
    category = category_names.get(record["category_id"], "Unknown")

    if country not in country_category_counts:
        country_category_counts[country] = {}
    if category not in country_category_counts[country]:
        country_category_counts[country][category] = 0
    country_category_counts[country][category] += 1

# Print the top 5 categories for each country (sorted, then sliced)
for country, categories in country_category_counts.items():
    top_5 = sorted(categories.items(), key=lambda item: item[1], reverse=True)[:5]
    print(f"\n{country} - top 5 categories")
    for category, count in top_5:
        print(f"  {category:<22}{count:>6,}")

print("\nDirect lookup example: US Music =", country_category_counts["US"]["Music"])


US - top 5 categories
  Entertainment          4,351
  Music                  3,039
  People & Blogs         1,419
  Comedy                 1,359
  Film & Animation         819

GB - top 5 categories
  Music                  4,990
  Entertainment          2,826
  People & Blogs         1,292
  Comedy                   937
  Film & Animation         710

IN - top 5 categories
  Entertainment          5,093
  Music                  2,281
  News & Politics        1,138
  People & Blogs         1,093
  Comedy                   962

Direct lookup example: US Music = 3039


## 5. Refactor — repeated block → reusable function

My first draft built the unique-ID set with the **same loop copied three times** (once each for US, GB and IN):

```python
us_video_ids = set()
for record in us_records:
    us_video_ids.add(record["video_id"])

gb_video_ids = set()
for record in gb_records:
    gb_video_ids.add(record["video_id"])

in_video_ids = set()
for record in in_records:
    in_video_ids.add(record["video_id"])
```

I replaced it with the function `unique_ids(records)` (defined in Section 4), called once per country inside a dictionary comprehension.

**What I gained:** the logic now lives in one place, so a fix or change is made once instead of three times, and adding a fourth country needs no new code, only another entry in `COUNTRIES`. It also makes the intent readable at a glance: `unique_ids(records_by_country[c])`.

## 6. Variable scope

- `project_name` is defined **outside** the function, so it is a **global** variable: code inside the function can read it, and so can code outside.
- `local_country` is defined **inside** `scope_demo()`, so it is a **local** variable: it exists only while the function runs. Trying to use it outside raises a `NameError`.
- The reverse is also shown: a function can *read* a global, but assigning to the same name inside a function creates a **new local** variable and leaves the global unchanged.

In [81]:
project_name = "TrendPulse"              # global: defined outside any function

def scope_demo():
    local_country = "US"                 # local: exists only inside this function
    print("Inside the function:")
    print("  global  project_name :", project_name)
    print("  local   local_country:", local_country)

    project_name_copy = project_name     # reading a global is allowed
    return project_name_copy

scope_demo()

print("\nOutside the function:")
print("  global  project_name :", project_name)
try:
    print("  local   local_country:", local_country)
except NameError as error:
    print("  local   local_country -> NameError:", error)

Inside the function:
  global  project_name : TrendPulse
  local   local_country: US

Outside the function:
  global  project_name : TrendPulse
  local   local_country -> NameError: name 'local_country' is not defined


In [82]:
# Assigning to a global's name inside a function creates a NEW local variable
def shadow_demo():
    project_name = "changed inside function"
    print("Inside  :", project_name)

shadow_demo()
print("Outside :", project_name, " <- global is unchanged")

Inside  : changed inside function
Outside : TrendPulse  <- global is unchanged


## 7. Loops over strings and sequences

`title_features(title)` loops over a title's **words** (counting fully-uppercase words) and over its **characters** with `range()` and index access (counting `!` and `?`). A word counts as "uppercase" only if it has **2+ letters**, so a lone `A` or `I` isn't counted as shouting. A missing title (`NaN`) is treated as an empty string so the function never crashes.

The `while` loop scans records until the first video above the view threshold: `continue` skips records with unknown views or a removed/error flag, and `break` stops as soon as the target is found. The index is advanced **before** `continue`, so the loop always terminates.

In [83]:
def title_features(title):
    """Count uppercase words, '!' and '?' marks, and words in a video title."""
    if not isinstance(title, str):          # missing titles arrive as NaN (a float)
        title = ""

    words = title.split()

    uppercase_words = 0
    for word in words:                                       # loop over WORDS
        letters = [ch for ch in word if ch.isalpha()]
        if len(letters) >= 2 and word.isupper():
            uppercase_words += 1

    exclamation_marks = 0
    question_marks = 0
    for i in range(len(title)):                              # range() + index access over CHARACTERS
        if title[i] == "!":
            exclamation_marks += 1
        elif title[i] == "?":
            question_marks += 1

    return {
        "uppercase_words": uppercase_words,
        "exclamation_marks": exclamation_marks,
        "question_marks": question_marks,
        "title_length_words": len(words),
    }

for example in ["Honest Football Prank LEAKED", "WHY?! This is INSANE!!!", "", None]:
    print(f"{example!r:<32}-> {title_features(example)}")

'Honest Football Prank LEAKED'  -> {'uppercase_words': 1, 'exclamation_marks': 0, 'question_marks': 0, 'title_length_words': 4}
'WHY?! This is INSANE!!!'       -> {'uppercase_words': 2, 'exclamation_marks': 4, 'question_marks': 1, 'title_length_words': 4}
''                              -> {'uppercase_words': 0, 'exclamation_marks': 0, 'question_marks': 0, 'title_length_words': 0}
None                            -> {'uppercase_words': 0, 'exclamation_marks': 0, 'question_marks': 0, 'title_length_words': 0}


In [84]:
# Apply it to the real data: average title length per country
for country in COUNTRIES:
    total_words = 0
    for record in records_by_country[country]:
        total_words += title_features(record["title"])["title_length_words"]
    average = total_words / len(records_by_country[country])
    print(f"{country}: average title length = {average:.2f} words")

US: average title length = 3.79 words


GB: average title length = 4.16 words
IN: average title length = 3.69 words


In [85]:
# while loop: scan until the first video above the view threshold
index = 0
scanned = 0
found = None

while index < len(cleaned_records):
    record = cleaned_records[index]
    index += 1                      # advance FIRST so 'continue' can never cause an infinite loop
    scanned += 1

    if record["views"] is None or record["video_error_or_removed"] is True:
        continue                    # skip records we cannot trust

    if record["views"] > VIRAL_VIEWS:
        found = record
        break                       # target found - stop scanning

if found:
    print(f"Scanned {scanned} record(s) before finding one above {VIRAL_VIEWS:,} views:")
    print(f"  {found['country']} | {found['title']} | {found['views']:,} views")
else:
    print("No record above the threshold.")

Scanned 8 record(s) before finding one above 1,000,000 views:
  US | Official Cats Speech EXCLUSIVE | 2,841,278 views


## 8. Rule-based classification

**Like ratio** = `likes / (likes + dislikes)`.

### Why these thresholds?
I checked the real data before choosing them: the **median like ratio is about 0.95** and 90% of videos are above 0.88. My first draft used 0.70, but only ~2% of videos fall below that, so it separated almost nothing and put ~74% of videos in one label. Thresholds that sit inside the real spread are far more informative.

| Setting | Value | Reason |
|---|---|---|
| `VIRAL_VIEWS` | 1,000,000 | about the top 20% of trending rows by views (the 90th percentile is ~2M) |
| `STEADY_VIEWS` | 100,000 | separates videos with a real audience from the long tail |
| `HEALTHY_RATIO` | 0.90 | about 1 in 7 rated videos falls below it, so it marks "noticeably less liked than most" |
| `DIVISIVE_RATIO` | 0.85 | below this, at least 1 vote in 7 was a dislike, and only about 6% of videos fall here |

### Labels (checked in this order)
1. **Unrated** — we cannot judge reception: ratings or comments disabled, video removed/errored, any of views/likes/dislikes/flags missing, or **zero likes and zero dislikes** (the ratio would divide by zero).
2. **Low Engagement** — zero views (edge case) *or* under 100,000 views with an acceptable ratio.
3. **Divisive** — like ratio **below 0.85**, regardless of views.
4. For ratios of 0.85 or higher, by views:
   - 1,000,000+ views and ratio **≥ 0.90** → **Viral Hit**
   - 1,000,000+ views and ratio **below 0.90** → **Steady Performer**
   - 100,000 to 999,999 views → **Steady Performer**
   - under 100,000 views → **Low Engagement**

Edge cases are handled **before** any division, so a division-by-zero is impossible.

In [86]:
def classify_video(record):
    """Return one label: Viral Hit, Steady Performer, Divisive, Low Engagement or Unrated."""
    views = record["views"]
    likes = record["likes"]
    dislikes = record["dislikes"]
    flags = (record["ratings_disabled"], record["comments_disabled"], record["video_error_or_removed"])

    # --- Edge cases first: missing values, disabled flags, removed videos ---
    if None in (views, likes, dislikes) or None in flags:
        return "Unrated"
    if record["ratings_disabled"] or record["comments_disabled"] or record["video_error_or_removed"]:
        return "Unrated"

    total_votes = likes + dislikes
    if total_votes == 0:                    # protects the division below
        return "Unrated"
    if views == 0:
        return "Low Engagement"

    like_ratio = likes / total_votes        # safe: total_votes > 0

    # --- Nested conditions ---
    if like_ratio < DIVISIVE_RATIO:
        return "Divisive"
    else:
        if views >= VIRAL_VIEWS:
            if like_ratio >= HEALTHY_RATIO:
                return "Viral Hit"
            else:
                return "Steady Performer"
        elif views >= STEADY_VIEWS:
            return "Steady Performer"
        else:
            return "Low Engagement"

### Hand-trace of one real record

**Record:** US — *"Official Cats Speech EXCLUSIVE"* — `views = 2,841,278`, `likes = 52,635`, `dislikes = 7,691`, all three flags `False`.

| Step | Condition | Result | Outcome |
|---|---|---|---|
| 1 | any of views/likes/dislikes/flags is `None`? | all present | continue |
| 2 | `ratings_disabled`, `comments_disabled` or `video_error_or_removed` true? | all `False` | continue |
| 3 | `total_votes = 52,635 + 7,691 = 60,326`; equal to 0? | no | continue |
| 4 | `views == 0`? | no | continue |
| 5 | `like_ratio = 52,635 / 60,326 = 0.8725` | | |
| 6 | `like_ratio < 0.85` (Divisive)? | 0.8725 < 0.85 is **False** | go to the `else` branch |
| 7 | `views >= 1,000,000`? | 2,841,278 is **True** | go into the inner block |
| 8 | `like_ratio >= 0.90`? | 0.8725 >= 0.90 is **False** | `else` branch |

**Label: Steady Performer** — hugely viewed, but its like ratio (0.8725) is below the 0.90 bar for a Viral Hit. The code cell below confirms the trace.

In [87]:
# Verify the hand-trace against the function
traced = next(r for r in cleaned_records
              if r["country"] == "US" and r["title"] == "Official Cats Speech EXCLUSIVE")
ratio = traced["likes"] / (traced["likes"] + traced["dislikes"])
print(f"views={traced['views']:,}  likes={traced['likes']:,}  dislikes={traced['dislikes']:,}  like_ratio={ratio:.4f}")
print("classify_video ->", classify_video(traced))
assert classify_video(traced) == "Steady Performer"

views=2,841,278  likes=52,635  dislikes=7,691  like_ratio=0.8725
classify_video -> Steady Performer


In [88]:
# Edge-case tests with hand-made records: every one must return a label without crashing
base = {"views": 500_000, "likes": 9_000, "dislikes": 500,
        "comments_disabled": False, "ratings_disabled": False, "video_error_or_removed": False}

tests = [
    ("normal mid-size video",          {}),
    ("zero views (votes present)",     {"views": 0}),
    ("zero likes AND zero dislikes",   {"likes": 0, "dislikes": 0}),
    ("ratings disabled",               {"ratings_disabled": True}),
    ("comments disabled",              {"comments_disabled": True}),
    ("video removed/error",            {"video_error_or_removed": True}),
    ("missing likes (None)",           {"likes": None}),
    ("missing flag (None)",            {"ratings_disabled": None}),
    ("zero likes, some dislikes",      {"likes": 0, "dislikes": 50}),
    ("exactly 1,000,000 views, 0.90",  {"views": 1_000_000, "likes": 90, "dislikes": 10}),
]

for description, changes in tests:
    test_record = {**base, **changes}
    print(f"{description:<34}-> {classify_video(test_record)}")

normal mid-size video             -> Steady Performer
zero views (votes present)        -> Low Engagement
zero likes AND zero dislikes      -> Unrated
ratings disabled                  -> Unrated
comments disabled                 -> Unrated
video removed/error               -> Unrated
missing likes (None)              -> Unrated
missing flag (None)               -> Unrated
zero likes, some dislikes         -> Divisive
exactly 1,000,000 views, 0.90     -> Viral Hit


In [89]:
# Classify every record and count each label
LABELS = ["Viral Hit", "Steady Performer", "Divisive", "Low Engagement", "Unrated"]
label_counts = {label: 0 for label in LABELS}
label_by_country = {c: {label: 0 for label in LABELS} for c in COUNTRIES}

for record in cleaned_records:
    label = classify_video(record)
    record["label"] = label
    label_counts[label] += 1
    label_by_country[record["country"]][label] += 1

total = len(cleaned_records)
print(f"{'Label':<20}{'Records':>10}{'Share':>9}")
print("-" * 39)
for label in LABELS:
    print(f"{label:<20}{label_counts[label]:>10,}{label_counts[label] / total:>9.1%}")
print("-" * 39)
print(f"{'Total':<20}{sum(label_counts.values()):>10,}")

print("\nBy country:")
print(f"{'Label':<20}" + "".join(f"{c:>9}" for c in COUNTRIES))
for label in LABELS:
    print(f"{label:<20}" + "".join(f"{label_by_country[c][label]:>9,}" for c in COUNTRIES))

Label                  Records    Share
---------------------------------------
Viral Hit                8,586    18.7%
Steady Performer        26,013    56.5%
Divisive                 1,880     4.1%
Low Engagement           7,853    17.1%
Unrated                  1,668     3.6%
---------------------------------------
Total                   46,000

By country:
Label                      US       GB       IN
Viral Hit               2,773    2,958    2,855
Steady Performer        9,050    8,203    8,760
Divisive                  544      461      875
Low Engagement          3,104    2,781    1,968
Unrated                   529      597      542


## 9. Summary

- Loaded **three countries** (US, GB, IN) with the count reported for each, comfortably above the 40,000-row requirement.
- Cleaned every field with `try/except`, reported blank vs bad values per field, kept records with unknown values as `None`, and dropped only exact duplicate rows.
- Used a list of dictionaries, tuples, sets (intersection / difference), a dictionary counter, slicing and a nested dictionary, each for the job it suits.
- Demonstrated variable scope, `for` / `while` / `range()` / `break` / `continue`, and a `title_features` function over characters and words.
- Labelled every record with a transparent rule-based `classify_video` (thresholds justified from the data, edge cases tested, hand-trace included).

**Limitations to keep in mind:** the labels are my own analytical categories, not YouTube's. Like ratio is high for almost every video here (median about 0.95), so the Divisive / Viral split rests on fairly small differences; and this copy of the data has anonymised-looking channel names, so channel-level findings describe patterns in the data rather than real-world creators.

**Next:** Part 2 builds on the cleaned records (`cleaned_records`) for NumPy and Pandas analysis.